In [ ]:
# Minsearch performs exact nearest neighbourhood, which compares the query vector against ALL vectors. That is not
# admissible in higher datasets. A better approximation is doing ANN (Approximate nearest neighbour) which narrows
# the sesarch to likely matches. SQLitesearch, additionally to persist data after the app has shutdown, it is able to perform ANN search.

In [7]:
# Carga del modelo
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="lsh",
    db_path="faq_vectors2.db"
)

# SQLitesearch provides three ANN modes: 
# - lsh (default) up to 100k vectors , random hyperplane projections, 
# - ivf  from 10 to 500k vectors, K-means clustering
# - hnsw: 10k-1M+ vedtors, proximity graph (highest recall)

In [4]:
# Cargamos los datos del faq para poder probar su vectorización
from ingest import load_faq_data
documents = load_faq_data();

In [8]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text);

# Se usa apra la barra de progreso.
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)


  0%|          | 0/27 [00:00<?, ?it/s]

In [9]:
vs_index.fit(vectors, documents)


In [10]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [11]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'data-engineering-zoomcamp',
  'section': 'G

In [12]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [13]:
# Cierre del indice
vs_index.close()


In [14]:
# Usar el índice ya ingerido
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="lsh",
    db_path="faq_vectors2.db"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

In [16]:
results

[{'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BOOTSTRAP_SERVERS_CONFIG` in the scripts located at `src/main/java/org/example/` (e.g., `JsonConsumer.java`, `JsonProducer.java`) is pointing to the correct server URL (e.g., `europe-west3` vs `europe-west2`).\n\n2. Verify that the cluster key and secrets are updated in `src/main/java/org/example/Secrets.java` (`KAFKA_CLUSTER_KEY` and `KAFKA_CLUSTER_SECRET`).',
  

In [17]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [18]:
vector_assistant.rag("the program has already begun, can I still sign up?")


'Yes, you can still join. You can start learning and submitting homework while the form is open, even if the course has already begun. If you want a certificate, make sure to submit your project before submissions close.'

In [19]:
vs_index.close()
